TASK 1


In [1]:
import random
suits = ['Hearts', 'Diamonds', 'Clubs', 'Spades']
ranks = ['2', '3', '4', '5', '6', '7', '8', '9', '10',
         'Jack', 'Queen', 'King', 'Ace']

deck = [(r, s) for s in suits for r in ranks]
trials = 100000
red = 0
heart_and_red = 0
face = 0
diamond_and_face = 0
spade_or_queen_and_face = 0
for _ in range(trials):
    rank, suit = random.choice(deck)

    if suit in ['Hearts', 'Diamonds']:
        red += 1
        if suit == 'Hearts':
            heart_and_red += 1
    if rank in ['Jack', 'Queen', 'King']:
        face += 1
        if suit == 'Diamonds':
            diamond_and_face += 1
        if suit == 'Spades' or rank == 'Queen':
            spade_or_queen_and_face += 1

print("P(Red):", red / trials)
print("P(Heart | Red):", heart_and_red / red)
print("P(Diamond | Face):", diamond_and_face / face)
print("P(Spade OR Queen | Face):", spade_or_queen_and_face / face)

P(Red): 0.50086
P(Heart | Red): 0.4973046360260352
P(Diamond | Face): 0.25295211730611183
P(Spade OR Queen | Face): 0.5034819845149011


TASK 2:

Intelligence ─┐
              ├──> Grade ───> Pass
StudyHours ───┤
Difficulty ───┘



In [6]:
!pip install pgmpy
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination

bn_model = DiscreteBayesianNetwork([
    ('Intelligence', 'Grade'),
    ('Study', 'Grade'),
    ('Difficulty', 'Grade'),
    ('Grade', 'Pass')
])

cpd_intel = TabularCPD(
    variable='Intelligence',
    variable_card=2,
    values=[[0.7], [0.3]]
)

cpd_study = TabularCPD(
    variable='Study',
    variable_card=2,
    values=[[0.6], [0.4]]
)

cpd_diff = TabularCPD(
    variable='Difficulty',
    variable_card=2,
    values=[[0.4], [0.6]]
)

cpd_grade = TabularCPD(
    variable='Grade',
    variable_card=3,
    values=[
        [0.88, 0.65, 0.75, 0.35, 0.60, 0.25, 0.45, 0.15],
        [0.10, 0.25, 0.18, 0.45, 0.30, 0.45, 0.35, 0.45],
        [0.02, 0.10, 0.07, 0.20, 0.10, 0.30, 0.20, 0.40]
    ],
    evidence=['Intelligence', 'Study', 'Difficulty'],
    evidence_card=[2, 2, 2]
)

cpd_pass = TabularCPD(
    variable='Pass',
    variable_card=2,
    values=[
        [0.95, 0.80, 0.50],
        [0.05, 0.20, 0.50]
    ],
    evidence=['Grade'],
    evidence_card=[3]
)


bn_model.add_cpds(cpd_intel, cpd_study, cpd_diff, cpd_grade, cpd_pass)
assert bn_model.check_model()

solver = VariableElimination(bn_model)

print("Probability of Passing given Study=Sufficient & Difficulty=Hard:")
res1 = solver.query(variables=['Pass'], evidence={'Study': 0, 'Difficulty': 0})
print(res1)

print("\nProbability of Intelligence given Pass=Yes:")
res2 = solver.query(variables=['Intelligence'], evidence={'Pass': 0})
print(res2)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 38.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 6.5 MB/s eta 0:00:00


/usr/local/lib/python3.12/dist-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in a future release. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


Probability of Passing given Study=Sufficient & Difficulty=Hard:
+---------+-------------+
| Pass    |   phi(Pass) |
+=========+=============+
| Pass(0) |      0.9062 |
+---------+-------------+
| Pass(1) |      0.0938 |
+---------+-------------+

Probability of Intelligence given Pass=Yes:
+-----------------+---------------------+
| Intelligence    |   phi(Intelligence) |
+=================+=====================+
| Intelligence(0) |              0.7235 |
+-----------------+---------------------+
| Intelligence(1) |              0.2765 |
+-----------------+---------------------+


TASK 3


In [8]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.factors.discrete import TabularCPD
from pgmpy.inference import VariableElimination
bn = DiscreteBayesianNetwork([
    ('Disease','Fever'),
    ('Disease','Cough'),
    ('Disease','Fatigue'),
    ('Disease','Chills')
])
d = TabularCPD('Disease',2,[[0.3],[0.7]])
f1 = TabularCPD('Fever',2,[
    [0.9,0.5],
    [0.1,0.5]
], evidence=['Disease'], evidence_card=[2])
c1 = TabularCPD('Cough',2,[
    [0.8,0.6],
    [0.2,0.4]
], evidence=['Disease'], evidence_card=[2])
f2 = TabularCPD('Fatigue',2,[
    [0.7,0.3],
    [0.3,0.7]
], evidence=['Disease'], evidence_card=[2])
c2 = TabularCPD('Chills',2,[
    [0.6,0.4],
    [0.4,0.6]
], evidence=['Disease'], evidence_card=[2])
bn.add_cpds(d,f1,c1,f2,c2)
bn.check_model()
inf = VariableElimination(bn)
print(inf.query(variables=['Disease'], evidence={'Fever':0,'Cough':0}))
print(inf.query(variables=['Disease'], evidence={'Fever':0,'Cough':0,'Chills':0}))
print(inf.query(variables=['Fatigue'], evidence={'Disease':0}))

+------------+----------------+
| Disease    |   phi(Disease) |
+============+================+
| Disease(0) |         0.5070 |
+------------+----------------+
| Disease(1) |         0.4930 |
+------------+----------------+
+------------+----------------+
| Disease    |   phi(Disease) |
+============+================+
| Disease(0) |         0.6067 |
+------------+----------------+
| Disease(1) |         0.3933 |
+------------+----------------+
+------------+----------------+
| Fatigue    |   phi(Fatigue) |
+============+================+
| Fatigue(0) |         0.7000 |
+------------+----------------+
| Fatigue(1) |         0.3000 |
+------------+----------------+


TASK 4


In [9]:
import numpy as np
weather_states = ["Sunny","Cloudy","Rainy"]
T = np.array([
    [0.65,0.25,0.10],
    [0.30,0.45,0.25],
    [0.20,0.35,0.45]
])
def run_chain(days=10):
    s = 0
    path = [weather_states[s]]
    rain_days = 0
    for i in range(days-1):
        s = np.random.choice(3, p=T[s])
        path.append(weather_states[s])
        if s == 2:
            rain_days += 1
    return path, rain_days
N = 10000
count_ok = 0
for i in range(N):
    seq, r = run_chain(10)
    if r >= 3:
        count_ok += 1
print("Probability of at least 3 rainy days:", count_ok/N)

Probability of at least 3 rainy days: 0.3127
